# Task 4 — Statistical Modeling & Risk-Based Pricing

1. **Severity model** — regress `TotalClaims` on policies *with* a claim.
2. **Claim probability model** — classify whether a policy has any claim.
3. **Risk-based premium** = `P(claim) * Predicted severity + expense + margin`.
4. **Interpretability** via SHAP.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_insurance_data, add_derived_metrics
from src.modeling import (
    engineer_features, build_preprocessor,
    evaluate_regressors, evaluate_classifiers, expected_premium,
)

df = add_derived_metrics(load_insurance_data('../data/insurance_data.csv'))
df = engineer_features(df)
df.shape

## 1. Feature lists

In [ ]:
NUMERIC_FEATURES = [c for c in [
    'SumInsured', 'CalculatedPremiumPerTerm', 'CustomValueEstimate',
    'CapitalOutstanding', 'Kilowatts', 'Cubiccapacity', 'Cylinders',
    'NumberOfDoors', 'VehicleAge', 'InsuredValueGap', 'PremiumPerInsured',
] if c in df.columns]

CATEGORICAL_FEATURES = [c for c in [
    'Province', 'PostalCode', 'Gender', 'MaritalStatus', 'VehicleType',
    'Make', 'Bodytype', 'CoverType', 'CoverCategory', 'CoverGroup',
    'TermFrequency', 'NewVehicle', 'AlarmImmobiliser', 'TrackingDevice',
] if c in df.columns]

pre = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)

## 2. Severity model — claimants only

In [ ]:
claimants = df[df['HasClaim'] == 1].copy()
X_sev = claimants[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_sev = claimants['TotalClaims']

sev_results, sev_models = evaluate_regressors(X_sev, y_sev, pre)
pd.DataFrame([r.__dict__ for r in sev_results]).drop(columns='model')

## 3. Claim probability model — full portfolio

In [ ]:
X_clf = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_clf = df['HasClaim']

clf_results, clf_models = evaluate_classifiers(X_clf, y_clf, pre)
pd.DataFrame([r.__dict__ for r in clf_results]).drop(columns='model')

## 4. Compose the risk-based premium

In [ ]:
best_clf = clf_models['XGBoost'] if 'XGBoost' in clf_models else clf_models['RandomForest']
best_sev = sev_models['XGBoost'] if 'XGBoost' in sev_models else sev_models['RandomForest']

p_claim = best_clf.predict_proba(X_clf)[:, 1]
sev_hat = best_sev.predict(X_clf)

df['PredictedPremium'] = expected_premium(
    p_claim, sev_hat, expense_loading=150.0, profit_margin=100.0,
)
df[['CalculatedPremiumPerTerm', 'PredictedPremium']].describe()

## 5. SHAP interpretability for the best regressor

In [ ]:
import shap

sample = X_sev.sample(min(2000, len(X_sev)), random_state=42)
transformed = best_sev.named_steps['pre'].transform(sample)
feature_names = best_sev.named_steps['pre'].get_feature_names_out()

explainer = shap.Explainer(best_sev.named_steps['model'])
shap_values = explainer(transformed)
shap.summary_plot(shap_values, features=transformed, feature_names=feature_names, max_display=15)

## 6. Business interpretation
Document, for each of the top 5–10 features:
- direction (does it increase or decrease predicted claims?),
- magnitude in Rand,
- pricing/marketing recommendation.